In [3]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import resend
import os
from pydantic import BaseModel

In [4]:
load_dotenv(override=True)

True

In [8]:
openai_api_key=os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_


In [9]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [10]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [13]:
deepseek_client=AsyncOpenAI(base_url=DEEPSEEK_BASE_URL,api_key=deepseek_api_key)
gemini_client=AsyncOpenAI(base_url=GEMINI_BASE_URL,api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client )
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash" , openai_client= gemini_client) 
llama3_3_model=OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client= groq_client)

In [ ]:
sales_agent1=Agent(name="DeepSeek Sales Agent", instructions=instructions1, model=deepseek_model )
sales_agent2=Agent(name="Gemini Sales Agent", instructions=instructions2, model=gemini_model)
sales_agent3 = Agent(name="Llama3.3 Sales Agent",instructions=instructions3,model=llama3_3_model)

In [15]:
description = "Write a cold sales email"

tool1= sales_agent1.as_tool(tool_name="sales_agent1",tool_description=description)
tool2=sales_agent2.as_tool(tool_name="sales_agent2",tool_description=description)
tool3=sales_agent3.as_tool(tool_name="sales_agent3",tool_description=description)

In [16]:
@function_tool
def send_html_email(subject: str, html_body: str) -> str:
    """Send out an email with the given subject and HTML body to all sales prospects """
    
    from_email = "Ed <onboarding@resend.dev>"
    to_email = "moumita.ray.soft8@gmail.com"
    RESEND_API_KEY = os.environ.get("RESEND_API_KEY")
    
    headers = {
        "Authorization": f"Bearer {RESEND_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "from": from_email,
        "to": [to_email],
        "subject": subject,
        "html": html_body
    }
    
    response = requests.post("https://api.resend.com/emails", json=payload, headers=headers)
    
    if response.status_code == 202:
        return "Email sent successfully"
    else:
        return f"Email failed to send due to {response.text}"

In [18]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer=Agent(name="Email subject writer", instructions=html_instructions,model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer",tool_description="Write a subject for a cold sales email" )

html_converter=Agent(name="HTML email body converter",instructions=html_instructions,model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")